# TRELLIS Image-to-3D — Backend Colab (GPU L4)
Chọn **Runtime -> Change runtime type -> L4 GPU**, sau đó chạy lần lượt 4 cell bên dưới. Cell 4 kiểm tra API bằng ảnh mẫu và có thể mất vài phút.


In [ ]:
# Cell 1: Kiểm tra GPU, clone repo của team & cài Miniconda
!nvidia-smi
!rm -rf /content/TRELLIS
!git clone https://github.com/truong-newbie/image-to-3d-web.git /content/TRELLIS
!rm -rf /content/miniconda3
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
!bash /tmp/miniconda.sh -b -p /content/miniconda3
!rm /tmp/miniconda.sh


In [ ]:
%%bash
# Cell 2: Cài đặt môi trường trellis chuẩn cho Colab L4 (Ubuntu 24.04)
set -e
cd /content/TRELLIS
rm -rf /tmp/extensions

source /content/miniconda3/etc/profile.d/conda.sh
export CONDA_ALWAYS_YES="true"

conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

conda remove -n trellis --all -y || true
conda create -n trellis python=3.10 -y
conda activate trellis

conda install -y -c nvidia/label/cuda-12.8.0 \
  "cuda-nvcc=12.8*" "cuda-cudart-dev=12.8*" "cuda-cccl=12.8*" \
  libcublas-dev libcusparse-dev libcusolver-dev libcurand-dev libcufft-dev

export CUDA_HOME=/content/miniconda3/envs/trellis
export PATH=$CUDA_HOME/bin:$PATH
export CPATH=$CUDA_HOME/include:$CUDA_HOME/targets/x86_64-linux/include:${CPATH:-}
export LIBRARY_PATH=$CUDA_HOME/lib:$CUDA_HOME/targets/x86_64-linux/lib:${LIBRARY_PATH:-}
export TORCH_CUDA_ARCH_LIST="8.9"
export NVCC_APPEND_FLAGS="-allow-unsupported-compiler"

pip install torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121
pip install xformers==0.0.27.post2 --index-url https://download.pytorch.org/whl/cu121
pip install kaolin -f https://nvidia-kaolin.s3.us-east-2.amazonaws.com/torch-2.4.0_cu121.html

. ./setup.sh --basic
pip install "transformers==4.49.0" spconv-cu120 fastapi uvicorn python-multipart

mkdir -p /tmp/extensions
git clone https://github.com/NVlabs/nvdiffrast.git /tmp/extensions/nvdiffrast
pip install --no-build-isolation /tmp/extensions/nvdiffrast

git clone --recurse-submodules https://github.com/JeffreyXiang/diffoctreerast.git /tmp/extensions/diffoctreerast
pip install --no-build-isolation /tmp/extensions/diffoctreerast

git clone https://github.com/autonomousvision/mip-splatting.git /tmp/extensions/mip-splatting
pip install --no-build-isolation /tmp/extensions/mip-splatting/submodules/diff-gaussian-rasterization/

echo "Cài đặt hoàn tất!"


In [ ]:
# Cell 3: Cập nhật mã nguồn, khởi động API và mở link tạm
import json
import os
import re
import subprocess
import time
import urllib.request
from pathlib import Path

subprocess.run(["git", "-C", "/content/TRELLIS", "pull", "--ff-only"], check=True)
revision = subprocess.check_output(
    ["git", "-C", "/content/TRELLIS", "rev-parse", "--short", "HEAD"], text=True
).strip()
print(f"Đang chạy mã nguồn commit {revision}", flush=True)

if not os.path.exists("/usr/local/bin/cloudflared"):
    subprocess.run(
        "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared",
        shell=True, check=True,
    )

subprocess.run("pkill -f 'uvicorn server:app' || true", shell=True)
subprocess.run("pkill -f 'cloudflared tunnel' || true", shell=True)

env = os.environ.copy()
env["ATTN_BACKEND"] = "xformers"
env["SPCONV_ALGO"] = "native"
api_log = open("/tmp/api.log", "w")
api_proc = subprocess.Popen(
    ["/content/miniconda3/envs/trellis/bin/uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="/content/TRELLIS/web_app/backend",
    env=env,
    stdout=api_log,
    stderr=subprocess.STDOUT,
)

print("Đang khởi động backend và tải model TRELLIS...", flush=True)
deadline = time.monotonic() + 900
while time.monotonic() < deadline:
    if api_proc.poll() is not None:
        raise RuntimeError("API dừng khi khởi động:\n" + Path("/tmp/api.log").read_text()[-4000:])
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=2) as response:
            health = json.load(response)
        if health.get("model_loaded"):
            print(f"Backend sẵn sàng trên GPU: {health.get('gpu')}", flush=True)
            break
    except Exception:
        pass
    time.sleep(2)
else:
    raise TimeoutError("API tải model quá 15 phút:\n" + Path("/tmp/api.log").read_text()[-4000:])

tunnel_log = open("/tmp/cloudflared.log", "w")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
)
for _ in range(60):
    if tunnel_proc.poll() is not None:
        raise RuntimeError("Cloudflare Tunnel đã dừng:\n" + Path("/tmp/cloudflared.log").read_text()[-4000:])
    match = re.search(
        r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
        Path("/tmp/cloudflared.log").read_text(),
    )
    if match:
        print("\n>>> LINK API CỦA BẠN:", match.group(0), flush=True)
        print("Giữ phiên Colab hoạt động trong lúc dùng Web UI. Chạy Cell 4 để kiểm tra API.", flush=True)
        break
    time.sleep(1)
else:
    raise TimeoutError("Không nhận được URL Cloudflare Tunnel:\n" + Path("/tmp/cloudflared.log").read_text()[-4000:])


In [ ]:
# Cell 4: Kiểm tra 1 ảnh, nhiều góc, hàng loạt và các định dạng tải xuống
!/content/miniconda3/envs/trellis/bin/python /content/TRELLIS/web_app/smoke_test.py --full
